In [8]:
from agents import Agent, function_tool, ModelSettings, RunContextWrapper

@function_tool
def weather_tool(city:str)->str:
    return f"The Weather in {city} is sunny."

agent=Agent(name="haiku agent",instructions="Always responde in haiku form",model="gpt-4.1-nano",tools=[weather_tool],model_settings=ModelSettings(tool_choice="weather_tool"))

In [2]:
# context
from dataclasses import dataclass
@dataclass
class UserContext:
    name:str
    uid:str
    is_pro_number:bool



In [3]:
# output_type
from pydantic import BaseModel
from agents import Agent

class CalendarEvent(BaseModel):
    name:str
    date:str
    participants:list[str]
agent=Agent(name="calendar extractor",instructions="Extract calendar events from text.",output_type=CalendarEvent)

In [5]:
def dynamic_instructions(context: RunContextWrapper[UserContext],agent:Agent[UserContext])->str:
    return f"The users name is {context.context.name}. Help them with their questions."

agent=Agent[UserContext](name="Triage agent", instructions=dynamic_instructions)

In [7]:
pirate_agent=Agent(name="pirate",instructions="write like a pirate",model="gpt-4.1-nano")

robot_agent=pirate_agent.clone(name="Robot",instructions="write like a robot")

In [9]:
# Tool Behavior
from agents import Agent, Runner, function_tool, ModelSettings

@function_tool
def weather_tool(city:str)->str:
    return f"The Weather in {city} is sunny."

agent=Agent(name="Weather Agent",
            instructions="Retrieve weather details.",
            model="gpt-4.1-nano",tools=[weather_tool],tool_use_behavior="stop_on_first_tool")


In [12]:
from agents.agent import StopAtTools, ToolsToFinalOutputResult
agent=Agent(name="Weather Agent",
            instructions="Retrieve weather details.",
            model="gpt-4.1-nano",tools=[weather_tool],tool_use_behavior=StopAtTools(stop_at_tool_names=['weather_tool']))

In [13]:
from typing import List, Any
from agents import FunctionToolResult
def custom_tool_handler(context: RunContextWrapper[Any], tool_results: List[FunctionToolResult])->ToolsToFinalOutputResult:
    for result in tool_results:
        if result.output and 'sunny' in result.output:
            return ToolsToFinalOutputResult(
                is_final_output=True,
                final_output=f"Final weather: {result.output}"
            )
    return ToolsToFinalOutputResult(is_final_output=False,final_output=None)
agent=Agent(name="Weather Agent",
            instructions="Retrieve weather details.",
            model="gpt-4.1-nano",tools=[weather_tool],tool_use_behavior=custom_tool_handler)

In [14]:
from agents import Agent, Runner

agent=Agent(name="Assistant", instructions="You are a helpful assistant")
result=await Runner.run(agent,"Write a haiku about recursion in programming.")
print(result.final_output)

Code calls itself back,  
Endless loops define the path—  
Depth in logic's grasp.


In [16]:
agent=Agent(name="Assistant", instructions="Reply very Concisely.")
thread_id="thread_123"
# with trace(workflow_name="Conversation",group_id=thread_id):
result=await Runner.run(agent,"what city is the golden gate bridge in?")
print(result.final_output)

new_inputs=result.to_input_list()+[{"role":"user", "content":"what state is it in?"}]
result=await Runner.run(agent,new_inputs)
print(result.final_output)


San Francisco.
California.


In [17]:
from agents import Agent, Runner, SQLiteSession
session = SQLiteSession("conversation_123")
result=await Runner.run(agent,"what city is the golden gate bridge in?",session=session)
print(result.final_output)

new_inputs=result.to_input_list()+[{"role":"user", "content":"what state is it in?"}]
result=await Runner.run(agent,new_inputs)
print(result.final_output)

San Francisco.
California.


In [3]:
from agents import Agent, Runner, SQLiteSession

agent=Agent(name="Assistant", instructions= "Reply very concisely.")

session=SQLiteSession("conversation_123")

result=await Runner.run(agent, "What city is the Golden Gate Bridge in?",session=session)

print(result.final_output)

result=await Runner.run(agent,"What state is it in?",session=session)

print(result.final_output)

result=await Runner.run(agent,"What's the population?",session=session)

print(result.final_output)


San Francisco.
California.
As of 2021, San Francisco's population is approximately 873,000.


In [5]:
from agents import SQLiteSession
session=SQLiteSession("user_123","conversations.db")

items=await session.get_items()

new_items=[
    {"role":"user","content":"Hello"},
    {"role":"assistant","content":"Hi there!"}
]
await session.add_items(new_items)

last_item=await session.pop_item()
print(last_item)
await session.clear_session()

{'role': 'assistant', 'content': 'Hi there!'}


In [11]:
from agents import Agent, Runner, SQLiteSession
agent=Agent(name="Assistant",model="gpt-4.1-nano")
session=SQLiteSession("Correction_example")

result=await Runner.run(agent,"What's 2+2 ?", session=session)
print(f"Agent: {result.final_output}")

Agent: 2 + 2 = 4


In [12]:
assistant_item=await session.pop_item()
user_item=await session.pop_item()
result=await Runner.run(agent,"What's 2+5 ?", session=session)
print(f"Agent: {result.final_output}")


Agent: 2 + 5 = 7


In [15]:
result=await Runner.run(agent,"hello")
print(result.final_output)

Hello! How can I assist you today?


In [16]:
session=SQLiteSession("user_123","conversations.db")
result=await Runner.run(agent,"hello",session=session)
print(result.final_output)

Hello! How can I assist you today?


In [17]:
session_1=SQLiteSession("user_123","conversations.db")
session_2=SQLiteSession("user_456","conversations.db")
result=await Runner.run(agent,"Hello",session=session_1)
print(result.final_output)
result=await Runner.run(agent,"Hello",session=session_2)
print(result.final_output)

Hello again! How can I help you today?
Hello! How can I assist you today?


In [22]:
from agents.memory import Session
from typing import List, Any, Optional
class CustomSession:
    def __init__(self,session_id:str)->str:
        self.session_id=session_id
        self.conversation_history:List[dict]=[]
    
    async def get_items(self, limit:int | None=None)->List[dict]:
        """Retrieve conversation history for this session."""
        if limit:
            return self.conversation_history[-limit:]
        return self.conversation_history
    async def add_items(self,items:List[dict])->None:
        """Store new items for this session."""
        return self.conversation_history.extend(items)
    async def pop_items(self)->Optional[dict]:
        """Remove and return the most recent item from this session."""
        if self.conversation_history:
            return self.conversation_history.pop()
        return None
    async def clear_session(self)->None:
        """Clear all items for this session."""
        return self.conversation_history.clear()

agent=Agent(name="Assistant",model="gpt-4.1-nano")
result=await Runner.run(agent,"Hello",session=CustomSession("my_session"))
print(result.final_output)



Hello! How can I assist you today?


In [23]:
await session.clear_session()
support_agent=Agent(name="Support",model="gpt-4.1-nano")
billing_agent=Agent(name="Billing",model="gpt-4.1-nano")
session=SQLiteSession("user_123")

result_1=await Runner.run(support_agent,"Help me with my account",session=session)
print(result_1.final_output)

result_2=await Runner.run(billing_agent,"what are my charges?",session=session)
print(result_2.final_output)

Of course! Could you please provide some more details about the issue you're experiencing with your account? This will help me assist you better.
I don’t have access to your account details or charges. To find out your charges, please check your account through the official app or website, or contact the customer support of your service provider directly. If you tell me which service or account you're referring to, I can guide you on how to check your charges.


In [38]:
# Tools
# 1. Hosted tools
import asyncio
from agents import Agent, FileSearchTool, WebSearchTool, Runner
agent=Agent(name="Assistant",
            tools=[WebSearchTool(),FileSearchTool(max_num_results=3,vector_store_ids=["VECTOR_STORE_ID"])])
async def main():
    result=await Runner.run(agent,"what is the weather in Malappuram and best cafe in edappal")
    print(result.final_output)


In [39]:
main()

<coroutine object main at 0x0000021A50502400>